In [1]:
# !python -m pip install --upgrade pip setuptools wheel

In [1]:


# !pip install ffmpeg_python==0.2.0 \
# imageio[ffmpeg]==2.37.0 \
# imagecodecs==2025.3.30 \
# gradio==5.22.0 \
# insightface==0.7.3 \ OR #!pip install --upgrade --user insightface
# numpy==1.26.4 \
# onnx==1.15.0 \
# onnxruntime_gpu==1.21.0 \
# opencv_python==4.8.1.78 \
# opencv_python_headless==4.8.1.78 \
# scikit-image==0.20.0 \
# tqdm==4.67.1 \
# psutil==7.0.0 \
# ngrok==1.4.0 \
# pyfiglet==1.0.2 \
# gdown==5.2.0 \
# lpips==0.1.4

In [2]:
# CELL 1: Re-download model with reliable 2026 mirror + verification
import os
import requests
from tqdm import tqdm

model_dir = os.path.expanduser(r'C:\Users\aditya\Downloads')
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'inswapper_128.onnx')
#model_path = os.path.join(model_dir, 'inswapper_128_fp16.onnx')
if os.path.exists(model_path) and os.path.getsize(model_path) > 500_000_000:
    print("✅ Model already good")
else:
    print("⬇️ Downloading inswapper_128.onnx (~554 MB, 1-3 min)...")
    # Reliable mirror (still active March 2026)
    url = "https://huggingface.co/Aitrepreneur/insightface/resolve/main/inswapper_128.onnx"
    #url = "https://huggingface.co/hacksider/deep-live-cam/resolve/main/inswapper_128_fp16.onnx"

    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))

    with open(model_path, "wb") as f, tqdm(
        desc="Download", total=total, unit="iB", unit_scale=True
    ) as bar:
        for chunk in response.iter_content(chunk_size=1024*1024):
            size = f.write(chunk)
            bar.update(size)

    print("✅ Download complete!")

print("Model size:", round(os.path.getsize(model_path)/(1024*1024), 1), "MB")
print("Location:", model_path)

✅ Model already good
Model size: 528.6 MB
Location: C:\Users\aditya\Downloads\inswapper_128.onnx


In [3]:
try:
    print("Checking FFMPEG is installed or not...")
    !ffmpeg -version
except Exception as e:
    print("FFMPEG Not installed....Install FFMPEG binaries based on your OS...")

Checking FFMPEG is installed or not...
ffmpeg version 2026-02-15-git-33b215d155-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
built with gcc 15.2.0 (Rev11, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdread --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopenjpeg --enable-libquirc --enable-libuavs3d --enable-libxevd --enable-libzvbi --enable-liboapv --enable-libqrencode --enable-librav1e --enable-libsvtav1 --enable-libvvenc --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxavs2 --enable-libx

In [4]:
%pwd

'C:\\Users\\aditya'

In [ ]:
# FOR LINUX BASED SYSTEMS

In [ ]:
# MODE 1 - Only inswapper128 used (128 full precision / fp16 BOTH CAN BER USED)
# CELL 2: Clean face-swap code (uses FULL path → works even if you edited the library)
import cv2
import insightface
from insightface.app import FaceAnalysis
import subprocess
import os

# GPU first, CPU fallback
try:
    app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
except Exception as e:
    app = FaceAnalysis()

app.prepare(ctx_id=0, det_size=(640, 640))

# FULL PATH fix (this is the key)
model_path = os.path.expanduser(r'C:\Users\aditya\Downloads\inswapper_128.onnx')
#model_path = os.path.expanduser('~/.insightface/models/inswapper_128.onnx')
#model_path = os.path.expanduser('~/.insightface/models/inswapper_128_fp16.onnx')
print("Loading swapper from:", model_path)
swapper = insightface.model_zoo.get_model(model_path)   # ← pass full path here

def swap_video(source_img_path, target_video_path, output_path):
    source_img = cv2.imread(source_img_path)
    source_faces = app.get(source_img)
    if len(source_faces) == 0:
        raise ValueError("No face detected in source image!")
    source_face = source_faces[0]

    cap = cv2.VideoCapture(target_video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('temp.mp4', fourcc, fps, (width, height))

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        faces = app.get(frame)
        for face in faces:
            frame = swapper.get(frame, face, source_face, paste_back=True)

        out.write(frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"✅ Processed {frame_count} frames")

    cap.release()
    out.release()

    # Re-encode with original audio
    subprocess.run([
        'ffmpeg', '-i', 'temp.mp4', '-i', target_video_path,
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '18',
        '-c:a', 'aac', '-map', '0:v:0', '-map', '1:a:0',
        '-shortest', '-y', output_path
    ], check=True)

    os.remove('temp.mp4')
    print(f"🎉 DONE! Video saved → {output_path}")

# Run it
# NOTE IF YOU GET ValueError: No face detected in source image!
# then check whether the given source image really have women face? If yes then face is too large, try to crop / small the facial region / or simply use different image of that women where her face is not too large i.e occupying the whole frame.

swap_video(r'C:\Users\aditya\Downloads\aisa.PNG', r'C:\Users\aditya\Documents\4Videosoft Studio\Video_260328072130_Slice.mp4', r'C:\Users\aditya\Documents\4Videosoft Studio\swapped_output.mp4')

In [5]:
# BELOW CODE WORKS ON WINDOWS

In [6]:
import onnxruntime as ort
if 'CUDAExecutionProvider' in list(ort.get_available_providers()):
    print(ort.get_available_providers())
    print("CUDA IS AVAILABLE FOR ONNX RUNTIME : GOOD")
else:
    print(ort.get_available_providers())
    print("CUDA IS NOT AVALIABLE, PROCESSING WILL BE DONE ON CPU")


['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
CUDA IS AVAILABLE FOR ONNX RUNTIME : GOOD


In [7]:
import cv2
import insightface
from insightface.app import FaceAnalysis
import subprocess
import os
import onnxruntime as ort

# -------------------------------
# GPU setup
# -------------------------------
print("Available ONNX runtime providers:", ort.get_available_providers())
try:
    app = FaceAnalysis(providers=['CUDAExecutionProvider'])
    print("✅ Using GPU (CUDA) if available")
except Exception as e:
    print("⚠️ CUDA not available, falling back to CPU")
    app = FaceAnalysis()
app.prepare(ctx_id=0, det_size=(640, 640))

# -------------------------------
# Load inswapper model
# -------------------------------
# Choose either FP32 or FP16 model
model_path = r'C:\Users\aditya\Downloads\inswapper_128.onnx'      # FP32
# model_path = r'C:\Users\aditya\Downloads\inswapper_128_fp16.onnx'  # FP16

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

print("Loading swapper from:", model_path)
swapper = insightface.model_zoo.get_model(model_path)

# -------------------------------
# Function to swap faces in video
# -------------------------------
def swap_video(source_img_path, target_video_path, output_path):
    # Load source image and detect face
    source_img = cv2.imread(source_img_path)
    if source_img is None:
        raise FileNotFoundError(f"Source image not found: {source_img_path}")

    source_faces = app.get(source_img)
    if len(source_faces) == 0:
        raise ValueError("No face detected in source image!")
    source_face = source_faces[0]

    # Open target video
    cap = cv2.VideoCapture(target_video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open target video: {target_video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # or 'XVID' if mp4v fails
    temp_video = 'temp_video.mp4'
    out = cv2.VideoWriter(temp_video, fourcc, fps, (width, height))

    frame_count = 0
    print("🔄 Starting face swap on video frames...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        try:
            faces = app.get(frame)
            for face in faces:
                frame = swapper.get(frame, face, source_face, paste_back=True)
        except Exception as e:
            print(f"⚠️ Frame {frame_count} failed to process: {e}")

        out.write(frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"✅ Processed {frame_count} frames")

    cap.release()
    out.release()

    # Re-encode with original audio using ffmpeg
    print("🎵 Merging original audio...")
    ffmpeg_cmd = [
        'ffmpeg', '-y', '-i', temp_video, '-i', target_video_path,
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '18',
        '-c:a', 'aac', '-map', '0:v:0', '-map', '1:a:0',
        '-shortest', output_path
    ]
    subprocess.run(ffmpeg_cmd, check=True)

    # Remove temporary video
    os.remove(temp_video)
    print(f"🎉 DONE! Swapped video saved to: {output_path}")

# -------------------------------
# Run the swap
# -------------------------------
swap_video(
    source_img_path=r'C:\Users\aditya\Downloads\aisa.PNG',
    target_video_path=r'C:\Users\aditya\Documents\4Videosoft Studio\Video_260328072130_Slice.mp4',
    output_path=r'C:\Users\aditya\Documents\4Videosoft Studio\swapped_output.mp4'
)

Available ONNX runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider

C:\Users\aditya\AppData\Roaming\Python\Python311\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


🔄 Starting face swap on video frames...


KeyboardInterrupt: 

In [1]:
# !pip uninstall onnxruntime onnxruntime-gpu -y
# !pip install onnxruntime-gpu

In [8]:
import cv2
import insightface
from insightface.app import FaceAnalysis
import subprocess
import os
import onnxruntime as ort

# -------------------------------
# GPU Check
# -------------------------------
if 'CUDAExecutionProvider' in ort.get_available_providers():
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    print("✅ CUDA is available for ONNX Runtime")
else:
    providers = ['CPUExecutionProvider']
    print("⚠️ CUDA not available, using CPU")

# -------------------------------
# FaceAnalysis GPU Setup
# -------------------------------
app = FaceAnalysis(providers=providers)
# ctx_id=0 forces GPU, det_size=640 for detection
app.prepare(ctx_id=0, det_size=(640, 640))
print("FaceAnalysis using providers:", app.providers)

# -------------------------------
# Load inswapper model
# -------------------------------
# Choose FP32 or FP16 model
model_path = r'C:\Users\aditya\Downloads\inswapper_128.onnx'      # FP32
# model_path = r'C:\Users\aditya\Downloads\inswapper_128_fp16.onnx'  # FP16

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

# Force GPU for swapper model
swapper = insightface.model_zoo.get_model(model_path, providers=providers)
print("Swapper using providers:", swapper._sess.get_providers())
print("Swapper shape:", swapper.input_shape)

# -------------------------------
# Function to swap faces in video
# -------------------------------
def swap_video(source_img_path, target_video_path, output_path):
    # Load source image
    source_img = cv2.imread(source_img_path)
    if source_img is None:
        raise FileNotFoundError(f"Source image not found: {source_img_path}")

    # Detect face in source image
    source_faces = app.get(source_img)
    if len(source_faces) == 0:
        raise ValueError("No face detected in source image!")
    source_face = source_faces[0]

    # Open target video
    cap = cv2.VideoCapture(target_video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open target video: {target_video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    temp_video = 'temp_video.mp4'
    out = cv2.VideoWriter(temp_video, fourcc, fps, (width, height))

    frame_count = 0
    print("🔄 Starting face swap on video frames...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        try:
            faces = app.get(frame)
            for face in faces:
                frame = swapper.get(frame, face, source_face, paste_back=True)
        except Exception as e:
            print(f"⚠️ Frame {frame_count} failed to process: {e}")

        out.write(frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"✅ Processed {frame_count} frames")

    cap.release()
    out.release()

    # Merge original audio using ffmpeg
    print("🎵 Merging original audio...")
    ffmpeg_cmd = [
        'ffmpeg', '-y', '-i', temp_video, '-i', target_video_path,
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '18',
        '-c:a', 'aac', '-map', '0:v:0', '-map', '1:a:0',
        '-shortest', output_path
    ]
    subprocess.run(ffmpeg_cmd, check=True)

    os.remove(temp_video)
    print(f"🎉 DONE! Swapped video saved to: {output_path}")

# -------------------------------
# Run the swap
# -------------------------------
swap_video(
    source_img_path=r'C:\Users\aditya\Downloads\aisa.PNG',
    target_video_path=r'C:\Users\aditya\Documents\4Videosoft Studio\Video_260328072130_Slice.mp4',
    output_path=r'C:\Users\aditya\Documents\4Videosoft Studio\swapped_output.mp4'
)

✅ CUDA is available for ONNX Runtime
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\aditya/.insightface\models\buffalo_l\w600k_r50.o

AttributeError: 'FaceAnalysis' object has no attribute 'providers'